# 🌱 Stemming with NLTK

**Big idea:** The same word shows up in many forms — `eat`, `eats`, `eating`. To a computer these are **3 different words**. **Stemming** chops off the endings (suffixes like `-ing`, `-s`, `-ed`) so they all become **one root form** called the **stem**.

> ✂️ Think of a gardener trimming branches: cut off the extra bits (endings) and keep the main stem.

**Why it matters:** fewer unique words ➜ smaller vocabulary ➜ simpler, faster models. A search for *"running"* can also match *"runs"*.

⚠️ **Catch:** stemming is a **rough rule-based cut**, not a dictionary lookup. The stem is **not always a real word** (`history` ➜ `histori`). That's okay — the computer just needs the forms to match.

### Step 1 — Our sample words
**Use case:** classifying product reviews as positive or negative. Reviews may say *eating*, *eats*, *eaten* — we want the model to treat them as the same idea: **eat**.

Below is a list of words in different forms that we'll stem with each tool.

In [20]:
## Classification Problem
## Comments of product is a positive review or negative review
## Reviews----> eating, eat,eaten [going,gone,goes]--->go

words=["eating","eats","eaten","writing","writes","programming","programs","history","finally","finalized"]

---
### Step 2 — `PorterStemmer`: the classic stemmer

**What:** The oldest and most famous stemming algorithm (Martin Porter, 1980). It applies a series of **suffix-removal rules** step by step.

- `from nltk.stem import PorterStemmer` → load the tool
- `PorterStemmer()` → create a stemmer object we can reuse
- `.stem(word)` → returns the stem of one word

In [21]:
from nltk.stem import PorterStemmer

In [22]:
stemming=PorterStemmer()

### Step 3 — Stem every word in our list

**Notice in the output:**
- `eating`, `eats` ➜ `eat` ✅
- `programming`, `programs` ➜ `program` ✅
- `finally`, `finalized` ➜ `final` ✅
- `eaten` ➜ `eaten` ❌ — irregular forms aren't handled (rules only cut endings, they don't "know" English)
- `history` ➜ `histori` ⚠️ — not a real word, but still a consistent stem

In [23]:
for word in words:
    print(word+" ----> "+stemming.stem(word))

eating ----> eat
eats ----> eat
eaten ----> eaten
writing ----> write
writes ----> write
programming ----> program
programs ----> program
history ----> histori
finally ----> final
finalized ----> final


### Step 4 — More examples
- `congratulations` ➜ `congratul` → aggressive cut, not a real word
- `sitting` ➜ `sit` → smartly removes the doubled `t` too ✅

> 🧠 Remember: **Porter** = rule-based suffix chopper. Fast, but sometimes ugly results.

In [24]:
stemming.stem('congratulations')

'congratul'

In [25]:
stemming.stem("sitting")

'sit'

---
### Step 5 — `RegexpStemmer`: build your own rules

**What:** You give it a **regular expression** (a text pattern), and it removes any part of the word that matches. Total control, no built-in grammar.

**Our pattern:** `'ing$|s$|e$|able$'`
- `|` means **OR**
- `$` means **"at the end of the word"**
- So: remove `ing` **or** `s` **or** `e` **or** `able` — only from the end.

`min=4` → words shorter than 4 letters are left alone (so `is` doesn't become `i`).

In [26]:
from nltk.stem import RegexpStemmer

In [27]:
reg_stemmer=RegexpStemmer('ing$|s$|e$|able$', min=4)

### Step 6 — Try it out

- `eating` ➜ `eat` ✅ (`ing` at the end removed)
- `ingeating` ➜ `ingeat` → only the **last** `ing` is removed, because of `$`. The `ing` at the start stays.

💡 Without `$` (pattern `'ing'`), it would remove **both** ➜ `eat`.

**When to use it:** quick custom cleanup for a specific domain where you know exactly which endings to strip.

> 🧠 Remember: **Regexp** = your rules, your responsibility.

In [28]:
reg_stemmer.stem('eating')

'eat'

In [29]:
reg_stemmer.stem('ingeating')

'ingeat'

---
### Step 7 — `SnowballStemmer`: the improved Porter (Porter2)

**What:** Created by the same author (Martin Porter) to fix weaknesses in the original. Also supports **many languages** — that's why we pass `'english'`.

👉 In practice, prefer **Snowball** over Porter.

In [30]:
from nltk.stem import SnowballStemmer

In [31]:
snowballsstemmer=SnowballStemmer('english')

### Step 8 — Stem the same word list
For these words the output is **identical** to Porter. The difference shows up on trickier words 👇

In [32]:
for word in words:
    print(word+" ----> "+snowballsstemmer.stem(word))

eating ----> eat
eats ----> eat
eaten ----> eaten
writing ----> write
writes ----> write
programming ----> program
programs ----> program
history ----> histori
finally ----> final
finalized ----> final


### Step 9 — Porter vs Snowball: where they differ

Words ending in `-ly`:

| Word | Porter | Snowball |
|---|---|---|
| `fairly` | `fairli` ❌ | `fair` ✅ |
| `sportingly` | `sportingli` ❌ | `sport` ✅ |

Snowball handles adverbs much better.

In [33]:
stemming.stem("fairly"),stemming.stem("sportingly")

('fairli', 'sportingli')

In [34]:
snowballsstemmer.stem("fairly"),snowballsstemmer.stem("sportingly")

('fair', 'sport')

### Step 10 — Where **both** fail: `goes`

Both give `goe` ❌. Stemmers only cut endings; they don't know that `goes`, `went`, `gone` all mean **go**.

👉 For that, we need **Lemmatization** (next notebook), which uses a real dictionary (WordNet) to return proper root words.

In [35]:
snowballsstemmer.stem('goes')

'goe'

In [36]:
stemming.stem('goes')

'goe'

---
## 📝 Quick recap (cheat sheet)

| Tool | How it works | Personality |
|---|---|---|
| `PorterStemmer` | Fixed suffix rules (1980) | Classic; sometimes ugly stems (`fairli`) |
| `RegexpStemmer` | Your own regex pattern | Full control; only as good as your rules |
| `SnowballStemmer` | Improved Porter rules, multi-language | Best default choice |

**Limits of stemming:**
- Stems may not be real words (`histori`, `congratul`)
- Irregular forms are missed (`eaten`, `goes`)

**Golden rule:** use stemming when **speed** matters more than perfect words; use **lemmatization** when you need real dictionary words.